In [1]:
import numpy as np

from numba import njit

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

import plotly.graph_objects as go

import fastplotlib as fpl

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),Apple M4,IntegratedGPU,Metal,


To silence this warning, use a fully namespaced name.


In [2]:
time = 1000
dt = 0.005
steps = int(time / dt)
tau_steps = int(1.5 / dt)

test_size = 0.2

node_grid_len = 2 # Then a 2 by 2 grid
grid_dim = 2
total_node = node_grid_len ** grid_dim
matrix_size = (total_node) * grid_dim

m_diag = np.random.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(np.random.uniform(0.1, 0.3, size=matrix_size))

t = np.linspace(0, time, steps)

u_val = 3
u = (t.astype(int) % 2) * 2 * u_val - u_val

In [ ]:
K = np.zeros((matrix_size, matrix_size))
k_vals = np.random.uniform(0.5, 8, total_node)
k_val_index = 0


for i in range(1, total_node + 1):
    # check left
    j = i - 1 # For example node_i = 2 node_j = 1
    if 1 <= j % node_grid_len < node_grid_len:
        # Now for spring 2 to 1
        # pos is for matrix position not nodal pos
        i_pos = i * 2 - 2 # for spring 2 we get index 2
        j_pos = j * 2 - 2 # for spring 1 we get index 0
        K[i_pos, i_pos] = k_vals[k_val_index]
        K[i_pos, j_pos] = -k_vals[k_val_index]
        K[j_pos, j_pos] = k_vals[k_val_index]
        K[j_pos, i_pos] = -k_vals[k_val_index]

    # check top
    j = i - node_grid_len
    if j > 0 and 0 <= j % node_grid_len < node_grid_len:
        i_pos = i * 2 - 2 + 1
        j_pos = j * 2 - 2 + 1
        K[i_pos, i_pos] = k_vals[k_val_index]
        K[i_pos, j_pos] = -k_vals[k_val_index]
        K[j_pos, j_pos] = k_vals[k_val_index]
        K[j_pos, i_pos] = -k_vals[k_val_index]

In [40]:
@njit
def run_simulation(steps, dt, u, M_INV, K, C, N):
    x = np.zeros((steps, N))
    v = np.zeros((steps, N))

    acc = np.zeros(N)

    u_force = np.zeros(N)

    for i in range(1, steps):
        u_force[0] = u[i - 1]
        u_force[1] = u[i - 1]

        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + u_force)

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        u_force_next = np.zeros(N)
        u_force_next[0] = u[i]

        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + u_force_next)

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [41]:
x, v = run_simulation(steps, dt, u, M_INV, K, DAMP, matrix_size)

In [42]:
x[1000:1010].T.round(1)

array([[-5. , -5. , -5. , -5. , -5. , -5. , -5. , -5. , -5. , -5. ],
       [-5.6, -5.6, -5.7, -5.7, -5.7, -5.7, -5.7, -5.7, -5.7, -5.7],
       [-2. , -2.1, -2.1, -2.1, -2.2, -2.2, -2.2, -2.3, -2.3, -2.4],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [-1.3, -1.3, -1.3, -1.3, -1.3, -1.3, -1.3, -1.3, -1.3, -1.3],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ]])

In [43]:
xx, yy = np.meshgrid(np.arange(node_grid_len), np.arange(node_grid_len))
x_init = xx.flatten()
y_init = yy.flatten()
dots_data = np.column_stack([x_init, y_init]).astype(np.float32)

hor = x[0::2]
ver = x[1::2]

fig = fpl.Figure()
dots_graphic = fig[0, 0].add_scatter(data=dots_data, sizes=15, colors="magenta")

frame_tracker = 0
frames = 5
max_frames = 50

def update_springs(canvas):
    global frame_tracker

    frame_tracker = (frame_tracker + frames) % steps
    print(f"Frame: {frame_tracker}/{steps}", end="\r")

    if frame_tracker >= max_frames:
        canvas.clear_animations()

    current_state = x[frame_tracker]
    hor = current_state[0::2]
    ver = current_state[1::2]

    print(
        f"Node 0 disp: ({hor[0]:.2f}, {ver[0]:.2f}) | Node 3 disp: ({hor[3]:.2f}, {ver[3]:.2f})"
    )

    dots_graphic.data[:, 0] = x_init + hor
    dots_graphic.data[:, 1] = y_init + ver

fig.add_animations(update_springs)
fig.show()

RFBOutputContext()

Node 0 disp: (-0.03, -0.01) | Node 3 disp: (0.00, 0.00)
Node 0 disp: (-0.06, -0.02) | Node 3 disp: (0.00, 0.00)
Node 0 disp: (-0.38, -0.16) | Node 3 disp: (0.00, 0.00)
Node 0 disp: (-0.47, -0.20) | Node 3 disp: (0.00, 0.00)
Node 0 disp: (-0.57, -0.24) | Node 3 disp: (0.00, 0.00)
